# Scraper performance

**Testbed.** Google Colab, CPU-only runtime, the data on Google Drive. The crawl needed several Colab
sessions (a free session ends after at most about 12 hours); every number below adds up all of them.

**How it is measured (auditable).** Every source is in the crawl's `logs/` folder:

| Metric | Method | File |
|---|---|---|
| wall time | wall clock per session, summed | `benchmark_sessions.jsonl` |
| time per paper | total wall time / papers on arXiv; per-paper download time from the receipts | `benchmark.json`, `papers.csv` |
| entry discovery time | time of the arXiv API calls, added up over sessions (saved in the discovery manifest) | `benchmark.json` |
| RAM | `psutil` resident set size of the process, sampled every 5 s (and every 2 s by the downloader) | `ram_timeseries.csv`, `performance.json` |
| disk | `du`-style walk of `data/`, `state/`, `logs/` at the end of each session, plus the peak of raw archives waiting in `/tmp` | `benchmark_sessions.jsonl` |

The whole system runs in one Python process (the reference thread included), so its RSS is the memory
footprint of the full system.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

import report_data as data

data.check()
bench, papers = data.benchmark(), data.papers()
on_arxiv = papers[papers["versions"] > 0]
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False})


def pct(value):
    return "–" if value is None else f"{value:.2f}%"

sessions, perf = data.sessions(), data.performance()

## Benchmark summary

In [ ]:
wall = bench["wall_seconds_total"]
rows = [
    ("Colab sessions", f"{bench['sessions']}"),
    ("Total wall time, end to end", f"{wall / 3600:,.2f} h"),
    ("Average time per paper (wall time / papers on arXiv)", f"{bench['avg_seconds_per_paper_end_to_end']:.2f} s"),
    ("Download + processing time per paper (median)", f"{bench['paper_download_seconds_median']:.2f} s"),
    ("Entry discovery time", f"{bench['discovery_seconds'] / 60:,.1f} min"),
    ("Peak RAM", f"{bench['ram_peak_mb']:,.0f} MB"),
    ("Average RAM", f"{bench['ram_avg_mb']:,.0f} MB"),
    ("Maximum disk storage required", f"{bench['max_disk_bytes'] / data.MB:,.1f} MB"),
    ("Final output size (data/raw)", f"{bench['final_output_bytes'] / data.MB:,.1f} MB"),
]
display(pd.DataFrame(rows, columns=["Metric", "Value"]).style.hide(axis="index"))

## Colab sessions

In [ ]:
table = pd.DataFrame({
    "start (UTC)": sessions["start"].dt.strftime("%Y-%m-%d %H:%M"),
    "wall time (h)": (sessions["wall_seconds"] / 3600).round(2),
    "peak RAM (MB)": sessions["ram_peak_mb"].round(0),
    "average RAM (MB)": sessions["ram_avg_mb"].round(0),
    "project on disk at end (MB)": (sessions["project_bytes"] / data.MB).round(1),
    "raw archives in /tmp, peak (MB)": (sessions["local_temp_peak_bytes"] / data.MB).round(1),
})
table.index = range(1, len(table) + 1)
table.index.name = "session"
display(table)

## RAM over time

In [ ]:
series = pd.DataFrame(perf.get("ram_timeseries", []), columns=["seconds", "mb"])
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(series["seconds"] / 3600, series["mb"], lw=0.8, color="#3b7a8f", label="RSS")
ax.axhline(bench["ram_peak_mb"], ls="--", color="#c96f53", label=f"peak {bench['ram_peak_mb']:,.0f} MB")
ax.axhline(bench["ram_avg_mb"], ls=":", color="#7a7a7a", label=f"average {bench['ram_avg_mb']:,.0f} MB")
ax.set_xlabel("Hours since the first session started (gaps: between sessions)")
ax.set_ylabel("MB")
ax.set_ylim(0, bench["ram_peak_mb"] * 1.2)
ax.legend(loc="lower right")
ax.set_title("Resident memory of the crawler")
plt.tight_layout()
plt.show()
display(Markdown(
    f"Memory stays flat: archives are streamed to disk in 1 MB blocks and only the kept text files are "
    f"read, so RAM does not grow with the corpus. Peak **{bench['ram_peak_mb']:,.0f} MB**, average "
    f"**{bench['ram_avg_mb']:,.0f} MB** over {bench['sessions']} session(s)."))

## Time per paper

In [ ]:
done = on_arxiv[on_arxiv["outcome"] != "incomplete"]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(done["seconds"].clip(upper=done["seconds"].quantile(0.99)), bins=60, color="#3b7a8f", alpha=0.85)
ax.axvline(done["seconds"].median(), ls="--", color="#c96f53", label=f"median {done['seconds'].median():.2f} s")
ax.set_xlabel("Time per paper, all versions (s, last bin: 99th percentile or more)")
ax.set_ylabel("Papers")
ax.legend()
ax.set_title("Download and processing time per paper")
plt.tight_layout()
plt.show()

download_seconds = perf.get("stages", {}).get("download", 0)
versions = bench["versions_processed"]
pace = (f"Over the download stage ({download_seconds / 3600:,.2f} h for {versions:,} versions) the crawler "
        f"processed **one version every {download_seconds / versions:.2f} s**, "
        f"**{download_seconds / versions / data.ARXIV_DELAY:.2f}×** the floor. ") if download_seconds and versions else ""
display(Markdown(
    "The time of a version runs from the moment the downloader starts on it (including the wait for its "
    "request slot) until its files are on Drive; a paper's time adds up its versions. A version that follows a "
    "long transfer gets its slot at once, so single values below the pacing interval are normal. "
    f"arXiv allows one request every {data.ARXIV_DELAY:g} s on one connection, so **{data.ARXIV_DELAY:g} s per "
    f"version** is the floor for any compliant scraper. {pace}Unpacking and the writes to Google Drive run in "
    "background threads and overlap the wait, so only archives that take longer than the interval to transfer "
    f"add time. Entry discovery took **{bench['discovery_seconds'] / 60:,.1f} min** (100 IDs per API call), and "
    "the references (100 papers per Semantic Scholar batch request) ran in parallel with the downloads."))

## Storage

In [ ]:
before_total = on_arxiv["size_before"].sum()
values = {"source archives, unpacked": before_total / data.MB,
          "maximum disk required": bench["max_disk_bytes"] / data.MB,
          "final output (data/raw)": bench["final_output_bytes"] / data.MB}
fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(list(values), list(values.values()), color=["#c96f53", "#7a7a7a", "#3b7a8f"])
ax.bar_label(bars, labels=[f"{v:,.0f} MB" for v in values.values()], padding=3)
ax.invert_yaxis()
ax.set_xlim(0, max(values.values()) * 1.25)
ax.set_xlabel("MB")
ax.set_title("Storage")
plt.tight_layout()
plt.show()
display(Markdown(
    f"The unpacked sources would take **{before_total / data.MB / 1024:,.1f} GB**; the scraper never stores them: "
    f"each archive is unpacked on the runtime's local disk, only `.tex`/`.bib` are copied to Drive, and the "
    f"archive is deleted. At most a few archives wait in `/tmp` at the same time "
    f"(peak **{sessions['local_temp_peak_bytes'].max() / data.MB:,.0f} MB**), so the maximum disk needed is "
    f"**{bench['max_disk_bytes'] / data.MB:,.0f} MB**, far below the Colab disk."))